# 03a — Tabela de harmonização LULC

Cria uma tabela longa de correspondência a partir do Excel de harmonização. Os códigos dentro de cada célula são separados por `;`.

In [ ]:
from pathlib import Path
import pandas as pd

import sys
sys.path.append("/code/scripts")
from lulc_utils import split_codes, harm_id, build_lookup

In [ ]:
area = "extremadura" #opções: "centro" ou "extremadura"

fld = '/code/data/raw/lulc' 
out = f'/code/data/processed/{area}/lulc/harmonized_tables'

excel_file = f'{fld}/nomen_lulc_harm.xlsx'
sheet = "Folha2"

Path(out).mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_excel(excel_file, sheet_name=sheet, dtype=str)
df.head()

In [ ]:
lookup = pd.concat([
    build_lookup(df, "COS_2007_2018 Código", "cos_2007_2018"),
    build_lookup(df, "COS_1995 Código", "cos_1995"),
    build_lookup(df, "SIOSE_CODIIGE", "siose"),
    build_lookup(df, "SIOSE_AR_ID_COBERTURA", "siose_ar"),
], ignore_index=True)

lookup.head()

In [ ]:
dup = lookup[lookup.duplicated(["product", "code_original"], keep=False)]

if len(dup) > 0:
    display(dup.sort_values(["product", "code_original"]))
    raise ValueError("Existem códigos repetidos dentro do mesmo produto.")

print("Sem códigos repetidos por produto.")
print(lookup.groupby("product")["code_original"].nunique())

In [ ]:
lookup_file = f"{out}/lookup_lulc_harmonizacao.csv"
lookup.to_csv(lookup_file,index=False)
print("Tabela guardada em:", lookup_file)